In [46]:
from collections import Counter
import re

INDIR_BIGRAM = {}
DIR_BIGRAM = {}

TOP_K = 20

In [47]:
# Read first file
data = []

with open("data/bigrams.txt", "rb") as f:
	for byte_line in f:
		try:
			line = byte_line.decode("utf-8")
			data.append(line.strip())
		except UnicodeDecodeError:
			pass
for line in data:
	occur, w1, w2 = line.split()
	occur = int(occur)
	w1, w2 = w1.lower(), w2.lower()

	if w1 not in DIR_BIGRAM.keys():
		DIR_BIGRAM[w1] = {}

	if w2 not in INDIR_BIGRAM.keys():
		INDIR_BIGRAM[w2] = {}

	DIR_BIGRAM[w1][w2] = occur
	INDIR_BIGRAM[w2][w1] = occur

In [48]:
# Read second file
data = []
with open("data/coca_all_links.txt", "rb") as f:
	for byte_line in f:
		try:
			line = byte_line.decode("utf-8")
			data.append(line.strip())
		except UnicodeDecodeError:
			pass

for line in data:
	line = line.split()
	occur, w1, w2 = int(line[0]), line[1].lower(), line[2].lower()

	if w1 not in DIR_BIGRAM.keys():
		DIR_BIGRAM[w1] = {}

	if w2 not in DIR_BIGRAM[w1].keys():
		DIR_BIGRAM[w1][w2] = 0

	DIR_BIGRAM[w1][w2] += occur

	if w2 not in INDIR_BIGRAM.keys():
		INDIR_BIGRAM[w2] = {}

	if w1 not in INDIR_BIGRAM[w2].keys():
		INDIR_BIGRAM[w2][w1] = 0

	INDIR_BIGRAM[w2][w1] += occur

In [49]:
def calc_probs(vocab):
	for key in vocab.keys():
		words = vocab[key].items()
		total = sum([num for _, num in words])
		words = list(map(lambda x: (x[0], x[1] / total), words))

		words.sort(key=lambda x: x[1], reverse=False)
		vocab[key] = words


calc_probs(DIR_BIGRAM)
calc_probs(INDIR_BIGRAM)

In [50]:
max(DIR_BIGRAM["a"], key=lambda x: x[1])

('lot', 0.021956376402660804)

In [51]:
def find_words(text):
	return re.findall(r'\w+', text.lower())


def P(word, ww):
	N = sum(ww.values())
	return ww[word] / N


def candidates(word, WORDS):
	return is_known([word], WORDS) or is_known(edits1(word), WORDS) or is_known(edits2(word), WORDS) or [word]


def is_known(words, WORDS):
	return set(w for w in words if w in WORDS)


def edits1(word):
	letters = 'abcdefghijklmnopqrstuvwxyz'
	splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
	deletes = [L + R[1:] for L, R in splits if R]
	transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
	replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
	inserts = [L + c + R for L, R in splits for c in letters]
	return set(deletes + transposes + replaces + inserts)


def edits2(word):
	return (e2 for e1 in edits1(word) for e2 in edits1(e1))


def norvig_solution(in_w):
	WORDS = Counter(find_words(open('big.txt').read()))
	return max(candidates(in_w, WORDS), key=lambda x: P(x, WORDS))

In [52]:
import numpy as np


def levenstein_distance(word1, word2):
	w1_len = len(word1)
	w2_len = len(word2)

	lev = np.zeros((w1_len + 1, w2_len + 1), dtype=int)

	lev[:, 0] = np.arange(w1_len + 1)
	lev[0, :] = np.arange(w2_len + 1)

	for i in range(1, w1_len + 1):
		for j in range(1, w2_len + 1):
			m = 0
			if word1[i - 1] != word2[j - 1]:
				m = 1
			lev[i, j] = min(lev[i - 1, j] + 1, lev[i - 1, j - 1] + m, lev[i - 1, j] + 1)

	return lev[w1_len - 1, w2_len - 1].item()

In [53]:
def look_forward(word, next_word):
	if next_word not in INDIR_BIGRAM:
		return []
	
	prev_words = INDIR_BIGRAM[next_word]

	max_candidates = min(len(INDIR_BIGRAM), TOP_K)

	candidates = prev_words[:max_candidates]
	candidates = [word for word, _ in candidates]
	return candidates

In [54]:
def look_behind(word, prev_word):
	if prev_word not in DIR_BIGRAM:
		return []
	
	next_words = DIR_BIGRAM[prev_word]

	max_candidates = min(len(DIR_BIGRAM), TOP_K)

	candidates = next_words[:max_candidates]
	candidates = [word for word, _ in candidates]
	return candidates

In [55]:
def correction(word, candidates):
	dists = [levenstein_distance(word, c) for c in candidates]
	c = list(zip(candidates, dists))
	c.sort(key=lambda x: x[1], reverse=False)
	if len(c) > 0:
		return c[0][0]
	return word

In [83]:
def text_correction(text):
	text = text.lower().split()

	output = []

	for idx, word in enumerate(text):
		candidates = []

		# Base case - Norvig candidates
		norvig_candidates = norvig_solution(word)
		candidates.append(norvig_candidates)

		# Look forward
		if idx != len(text) - 1:
			forward_candidates = look_forward(word, text[idx + 1])
			candidates.extend(forward_candidates)

		# Look Behind
		if idx != 0:
			behind_candidates = look_behind(word, text[idx - 1])
			candidates.extend(behind_candidates)

		# TODO consider levenstein distance
		corrected = correction(word, candidates)
		output.append(corrected)

	return " ".join(output)

In [91]:
def test(text):
	print("Original text:", text)
	output = text_correction(text)
	print("Corrected text:", output, end='\n\n')

In [96]:
test("For tha sake of god")

test("I drave my cur")

Original text: For tha sake of god
Corrected text: for the sake of god

Original text: I drave my cur
Corrected text: i drove my our

